## **1. Project Overview**

The goal of this project is to analyze A/B test results using Python and statistical methods.

The project focuses on four conversion metrics:

- add_payment_info / session
- add_shipping_info / session
- begin_checkout / session
- new_accounts / session

The final output is a structured results table that can be used for further visualization and dashboard creation.

## **2. Environment Setup**

This section imports all required libraries for data extraction, transformation, statistical testing, and visualization.

In [ ]:
# ===============================
# Data processing
# ===============================
import pandas as pd
import numpy as np

# ===============================
# Visualization
# ===============================
import matplotlib.pyplot as plt

# ===============================
# Statistical testing
# ===============================
from statsmodels.stats.proportion import proportions_ztest

# ===============================
# Google BigQuery
# ===============================
from google.cloud import bigquery

# ===============================
# Display settings
# ===============================
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

## 3. Google BigQuery Connection

In this section, we authenticate in Google Colab and initialize the BigQuery client.

In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "data-analytics-mate"
client = bigquery.Client(project="data-analytics-mate")

print("Connected project:", client.project)

Connected project: data-analytics-mate


## **4. Data Extraction**

In this section, we extract A/B testing data from Google BigQuery.

The dataset contains daily values of sessions, selected funnel events, and new account creation across multiple dimensions such as country, device, continent, traffic channel, test, and test group.

This dataset will be used to calculate conversion metrics and statistical significance in Python.

In [ ]:
query = """
WITH session_info AS (
    SELECT
        s.date,
        s.ga_session_id,
        sp.country,
        sp.device,
        sp.continent,
        sp.channel,
        ab.test,
        ab.test_group
    FROM `data-analytics-mate.DA.ab_test` ab
    JOIN `data-analytics-mate.DA.session` s
        ON ab.ga_session_id = s.ga_session_id
    JOIN `data-analytics-mate.DA.session_params` sp
        ON sp.ga_session_id = ab.ga_session_id
),

sessions AS (
    SELECT
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group,
        'session' AS event_name,
        COUNT(DISTINCT session_info.ga_session_id) AS value
    FROM session_info
    GROUP BY
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group
),

events AS (
    SELECT
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group,
        ep.event_name,
        COUNT(ep.ga_session_id) AS value
    FROM `data-analytics-mate.DA.event_params` ep
    JOIN session_info
        ON ep.ga_session_id = session_info.ga_session_id
    WHERE ep.event_name IN (
        'add_payment_info',
        'add_shipping_info',
        'begin_checkout'
    )
    GROUP BY
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group,
        ep.event_name
),

new_accounts AS (
    SELECT
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group,
        'new_accounts' AS event_name,
        COUNT(DISTINCT acs.ga_session_id) AS value
    FROM `data-analytics-mate.DA.account_session` acs
    JOIN session_info
        ON acs.ga_session_id = session_info.ga_session_id
    GROUP BY
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group
)

SELECT * FROM sessions
UNION ALL
SELECT * FROM events
UNION ALL
SELECT * FROM new_accounts
"""

In [ ]:
df = client.query(query).to_dataframe()

print("Shape:", df.shape)
display(df.head())

Shape: (161567, 9)


,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-12-01,Guatemala,desktop,Americas,Direct,3,2,begin_checkout,2
1,2020-12-08,Guatemala,desktop,Americas,Paid Search,3,2,begin_checkout,8
2,2020-12-10,Lebanon,desktop,Asia,Organic Search,3,2,begin_checkout,2
3,2020-12-13,Uruguay,desktop,Americas,Social Search,3,2,add_shipping_info,5
4,2020-12-16,Lithuania,mobile,Europe,Direct,4,1,begin_checkout,20


## **5. Dataset Validation and Structure Check**

Before performing statistical analysis, we validate the extracted dataset.

The goal of this step is to:
- understand the dataset structure
- verify available tests and test groups
- check the available event types
- detect possible missing values or anomalies

This validation ensures that further calculations of conversion metrics and statistical significance are based on consistent and reliable data.

### **5.1 Dataset overview**

In [ ]:
print("Dataset shape:")
print(df.shape)

print("\nColumns:")
print(df.columns)

display(df.head())

Dataset shape:
(161567, 9)

Columns:
Index(['date', 'country', 'device', 'continent', 'channel', 'test',
       'test_group', 'event_name', 'value'],
      dtype='object')


,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-12-01,Guatemala,desktop,Americas,Direct,3,2,begin_checkout,2
1,2020-12-08,Guatemala,desktop,Americas,Paid Search,3,2,begin_checkout,8
2,2020-12-10,Lebanon,desktop,Asia,Organic Search,3,2,begin_checkout,2
3,2020-12-13,Uruguay,desktop,Americas,Social Search,3,2,add_shipping_info,5
4,2020-12-16,Lithuania,mobile,Europe,Direct,4,1,begin_checkout,20


### **5.2 Check available events**

In [ ]:
print("Available events in dataset:")

display(
    df["event_name"]
    .value_counts()
)

Available events in dataset:


,count
event_name,
session,107210
new_accounts,22389
add_shipping_info,11961
begin_checkout,11959
add_payment_info,8048


### **Available Event Types**

The dataset contains five key event types representing different stages of the user conversion funnel.

Event distribution:

- session — total number of user sessions
- new_accounts — sessions where users created a new account
- add_shipping_info — sessions where users entered shipping information
- begin_checkout — sessions where users started the checkout process
- add_payment_info — sessions where users entered payment details

The funnel structure follows a typical e-commerce conversion path, where the number of events decreases at each subsequent step. This confirms the logical consistency of the dataset and allows us to construct conversion metrics based on these events.

### **5.3 Check A/B tests**

In [ ]:
display(
    df["test"]
    .value_counts()
    .sort_index()
)

,count
test,
1,29114
2,32430
3,40827
4,59196


### **A/B Tests Overview**

The dataset contains results for multiple A/B experiments.

Four different tests are present in the dataset:

- Test 1
- Test 2
- Test 3
- Test 4

Each test contains two experiment groups (control and variant), which will be compared using statistical significance testing.

This structure allows us to evaluate whether product changes had a statistically significant impact on key conversion metrics.

### **5.4 Check test groups**

In [ ]:
print("Test groups:")

display(
    df["test_group"]
    .unique()
)

Test groups:


<IntegerArray>
[2, 1]
Length: 2, dtype: Int64

### **Test Groups**

The dataset contains two experiment groups:

- Group 1 — Control
- Group 2 — Variant

The control group represents the original product experience, while the variant group represents the modified version being tested.

Statistical significance testing will compare conversion rates between these two groups.

### **5.5 Check dimensions**

In [ ]:
print("Countries:", df["country"].nunique())
print("Devices:", df["device"].nunique())
print("Continents:", df["continent"].nunique())
print("Channels:", df["channel"].nunique())


Countries: 108
Devices: 3
Continents: 6
Channels: 5


### **Dataset Dimensions**

The dataset contains a wide range of dimensions that allow deeper analysis of A/B test results.

Key dimensions:

- 108 countries
- 3 device types
- 6 continents
- 5 traffic channels

This structure enables analysis not only at the total test level, but also across different segments such as geographic regions, devices, and marketing channels.

### **5.6 Check missing values**

In [ ]:
print("Missing values:")

display(
    df.isna().sum()
)

Missing values:


,0
date,0
country,0
device,0
continent,0
channel,0
test,0
test_group,0
event_name,0
value,0


### **Missing Values Check**

The dataset was checked for missing values across all columns.

No missing values were detected in any of the fields:

- date
- country
- device
- continent
- channel
- test
- test_group
- event_name
- value

This confirms that the dataset is complete and suitable for further statistical analysis without requiring additional data cleaning.

### **5.7 Check total event volumes**

In [ ]:
event_totals = (
    df.groupby("event_name")["value"]
    .sum()
    .sort_values(ascending=False)
)

display(event_totals)

,value
event_name,
session,542142
begin_checkout,59998
new_accounts,45202
add_shipping_info,33815
add_payment_info,23622


### **Event Volume Validation**

To validate the dataset, we examined the total volume of each event type.

Event totals:

- session — 542,142
- begin_checkout — 59,998
- add_shipping_info — 33,815
- add_payment_info — 23,622
- new_accounts — 45,202

The event counts follow a logical funnel structure where the number of events decreases as users progress through the checkout process.

This confirms that the dataset reflects a realistic user journey and is suitable for calculating conversion metrics.

## **6. Metric Definition and Data Preparation**

In this section, we prepare the dataset for A/B testing analysis.

We define four key conversion metrics that represent important steps in the user funnel:

- add_payment_info / session
- add_shipping_info / session
- begin_checkout / session
- new_accounts / session

Each metric is calculated as the ratio between a specific event and the number of sessions.

To ensure scalability, metrics are defined using a dictionary structure, allowing the analysis to automatically iterate through all metrics without hardcoding calculations.

### **6.1 Define metrics dictionary**

In [ ]:
# --------------------------------------------
# Define conversion metrics
# --------------------------------------------

metrics = {
    "add_payment_info": "session",
    "add_shipping_info": "session",
    "begin_checkout": "session",
    "new_accounts": "session"
}

print("Defined metrics:")
metrics

Defined metrics:


{'add_payment_info': 'session',
 'add_shipping_info': 'session',
 'begin_checkout': 'session',
 'new_accounts': 'session'}

### **6.2 Pivot dataset**

In [ ]:
# --------------------------------------------
# Pivot dataset to wide format
# --------------------------------------------

pivot_df = (
    df
    .pivot_table(
        index=[
            "date",
            "country",
            "device",
            "continent",
            "channel",
            "test",
            "test_group"
        ],
        columns="event_name",
        values="value",
        aggfunc="sum"
    )
    .reset_index()
)

# --------------------------------------------
# Fill missing values only in event columns
# --------------------------------------------

index_columns = [
    "date",
    "country",
    "device",
    "continent",
    "channel",
    "test",
    "test_group"
]

event_columns = [col for col in pivot_df.columns if col not in index_columns]

pivot_df[event_columns] = pivot_df[event_columns].fillna(0)

display(pivot_df.head())

event_name,date,country,device,continent,channel,test,test_group,add_payment_info,add_shipping_info,begin_checkout,new_accounts,session
0,2020-11-01,(not set),desktop,Africa,Paid Search,1,1,0,0,0,0,1
1,2020-11-01,(not set),desktop,Africa,Paid Search,2,1,0,0,0,0,1
2,2020-11-01,(not set),desktop,Americas,Direct,1,2,0,0,0,1,1
3,2020-11-01,(not set),desktop,Americas,Direct,2,1,0,0,0,1,1
4,2020-11-01,(not set),desktop,Americas,Paid Search,1,1,0,1,1,0,2


After pivoting the dataset to wide format, missing values are replaced with zeros only for event columns.

This is necessary because not every event occurs for every combination of dimensions, while dimension columns such as date and country must remain unchanged.

### **6.3 Check resulting structure**

In [ ]:
print("Columns after pivot:")
display(pivot_df.columns)

Columns after pivot:


Index(['date', 'country', 'device', 'continent', 'channel', 'test',
       'test_group', 'add_payment_info', 'add_shipping_info', 'begin_checkout',
       'new_accounts', 'session'],
      dtype='object', name='event_name')

### **Pivoted Dataset Structure**

After applying the pivot transformation, each event type is represented as a separate column.

This structure enables direct calculation of conversion metrics such as:

- add_payment_info / session
- add_shipping_info / session
- begin_checkout / session
- new_accounts / session

Each row now represents a unique combination of:

- date
- country
- device
- continent
- channel
- test
- test_group

This format is optimal for performing statistical comparisons between control and variant groups.

### **6.4 Validate metrics availability**

In [ ]:
print("Checking metric availability:")

for metric in metrics.keys():
    if metric in pivot_df.columns:
        print(f"{metric} ✔ available")
    else:
        print(f"{metric} ❌ missing")

Checking metric availability:
add_payment_info ✔ available
add_shipping_info ✔ available
begin_checkout ✔ available
new_accounts ✔ available


### **Metric Availability Check**

All required event types are present in the pivoted dataset.

The following events are available for metric calculation:

- add_payment_info
- add_shipping_info
- begin_checkout
- new_accounts

This confirms that all defined conversion metrics can be calculated correctly using the prepared dataset.

### **6.5 Example metric calculation**

In [ ]:
# Example conversion calculation
pivot_df["begin_checkout_rate"] = (
    pivot_df["begin_checkout"] / pivot_df["session"]
)

display(
    pivot_df[[
        "test",
        "test_group",
        "session",
        "begin_checkout",
        "begin_checkout_rate"
    ]].head()
)

event_name,test,test_group,session,begin_checkout,begin_checkout_rate
0,1,1,1,0,0.0000
1,2,1,1,0,0.0000
2,1,2,1,0,0.0000
3,2,1,1,0,0.0000
4,1,1,2,1,0.5000


### **Data Preparation Summary**

The dataset has been transformed into a wide format using a pivot table, where each event type is represented as a separate column.

This structure enables direct calculation of conversion rates and simplifies statistical testing.

All required metrics are present in the dataset and can now be processed dynamically using loops.

## 7. Statistical Significance Calculation

In this section, we calculate statistical significance for each conversion metric across all A/B tests.

The calculation is implemented dynamically using loops, which makes the approach scalable and suitable for any number of metrics.

For each test and metric, we calculate:

- control event count
- control session count
- control conversion rate
- variant event count
- variant session count
- variant conversion rate
- metric change (%)
- z-statistic
- p-value
- statistical significance

In [ ]:
# --------------------------------------------
# Define control and variant groups
# --------------------------------------------

CONTROL_GROUP = 1
VARIANT_GROUP = 2

# --------------------------------------------
# Initialize results container
# --------------------------------------------
results = []

# --------------------------------------------
# Loop through each test
# --------------------------------------------
for test_number in sorted(pivot_df["test"].dropna().unique()):

    test_data = pivot_df[pivot_df["test"] == test_number].copy()

    # Split into control and variant
    control_data = test_data[test_data["test_group"] == CONTROL_GROUP]
    variant_data = test_data[test_data["test_group"] == VARIANT_GROUP]

    # ----------------------------------------
    # Loop through each metric
    # ----------------------------------------
    for numerator_event, denominator_event in metrics.items():

        # Aggregate numerator and denominator for control
        numerator_control = control_data[numerator_event].sum()
        denominator_control = control_data[denominator_event].sum()

        # Aggregate numerator and denominator for variant
        numerator_variant = variant_data[numerator_event].sum()
        denominator_variant = variant_data[denominator_event].sum()

        # Skip invalid cases
        if denominator_control == 0 or denominator_variant == 0:
            continue

        # Conversion rates
        conversion_rate_control = numerator_control / denominator_control
        conversion_rate_variant = numerator_variant / denominator_variant

        # Relative change (%)
        # Example: if control = 4.0%, variant = 5.0%, change = +25%
        if conversion_rate_control == 0:
            metric_change = np.nan
        else:
            metric_change = (
                (conversion_rate_variant - conversion_rate_control)
                / conversion_rate_control
            ) * 100

        # Two-proportion z-test
        counts = [numerator_variant, numerator_control]
        nobs = [denominator_variant, denominator_control]

        z_stat, p_value = proportions_ztest(counts, nobs)

        # Significance flag
        significant = p_value < 0.05

        # Save result
        results.append({
            "test_number": test_number,
            "metric": numerator_event,
            "numerator_event": numerator_event,
            "denominator_event": denominator_event,
            "numerator_control": numerator_control,
            "denominator_control": denominator_control,
            "conversion_rate_control": conversion_rate_control,
            "numerator_variant": numerator_variant,
            "denominator_variant": denominator_variant,
            "conversion_rate_variant": conversion_rate_variant,
            "metric_change": metric_change,
            "z_stat": z_stat,
            "p_value": p_value,
            "significant": significant
        })

# --------------------------------------------
# Convert results to DataFrame
# --------------------------------------------
results_df = pd.DataFrame(results)

# --------------------------------------------
# Format numeric columns
# --------------------------------------------
results_df["conversion_rate_control"] = results_df["conversion_rate_control"].round(4)
results_df["conversion_rate_variant"] = results_df["conversion_rate_variant"].round(4)
results_df["metric_change"] = results_df["metric_change"].round(4)
results_df["z_stat"] = results_df["z_stat"].round(4)
results_df["p_value"] = results_df["p_value"].round(4)

# --------------------------------------------
# Sort results
# --------------------------------------------
results_df = results_df.sort_values(
    by=["test_number", "metric"]
).reset_index(drop=True)

display(results_df)



,test_number,metric,numerator_event,denominator_event,numerator_control,denominator_control,conversion_rate_control,numerator_variant,denominator_variant,conversion_rate_variant,metric_change,z_stat,p_value,significant
0,1,add_payment_info,add_payment_info,session,1988,45362,0.0438,2229,45193,0.0493,12.5420,3.9249,0.0001,True
1,1,add_shipping_info,add_shipping_info,session,3034,45362,0.0669,3221,45193,0.0713,6.5605,2.6036,0.0092,True
2,1,begin_checkout,begin_checkout,session,3784,45362,0.0834,4021,45193,0.0890,6.6606,2.9788,0.0029,True
3,1,new_accounts,new_accounts,session,3823,45362,0.0843,3681,45193,0.0815,-3.3543,-1.5429,0.1229,False
4,2,add_payment_info,add_payment_info,session,2344,50637,0.0463,2409,50244,0.0479,3.5769,1.2410,0.2146,False
5,2,add_shipping_info,add_shipping_info,session,3480,50637,0.0687,3510,50244,0.0699,1.6510,0.7096,0.4780,False
6,2,begin_checkout,begin_checkout,session,4262,50637,0.0842,4313,50244,0.0858,1.9882,0.9529,0.3406,False
7,2,new_accounts,new_accounts,session,4165,50637,0.0823,4184,50244,0.0833,1.2419,0.5888,0.5560,False
8,3,add_payment_info,add_payment_info,session,3623,70047,0.0517,3697,70439,0.0525,1.4746,0.6432,0.5201,False
9,3,add_shipping_info,add_shipping_info,session,5298,70047,0.0756,5188,70439,0.0737,-2.6212,-1.4137,0.1574,False


### **Statistical significance results**

The final results table contains conversion rates for control and variant groups, relative metric change, z-statistics, p-values, and significance flags for each metric across all tests.

This table is fully prepared for further visualization in Python or Tableau.

## **8. Preparing Results for Visualization**

In this section we finalize the results of statistical calculations and prepare a clean dataset for visualization.

The dataset includes:

- conversion rates for control and variant groups
- relative metric change
- z-statistic
- p-value
- statistical significance flag

This table will be exported and used to build a dashboard in Tableau Public.

In [ ]:
# --------------------------------------------
# Convert results list to DataFrame
# --------------------------------------------

results_df = pd.DataFrame(results)

display(results_df.head())

,test_number,metric,numerator_event,denominator_event,numerator_control,denominator_control,conversion_rate_control,numerator_variant,denominator_variant,conversion_rate_variant,metric_change,z_stat,p_value,significant
0,1,add_payment_info,add_payment_info,session,1988,45362,0.0438,2229,45193,0.0493,12.5420,3.9249,0.0001,True
1,1,add_shipping_info,add_shipping_info,session,3034,45362,0.0669,3221,45193,0.0713,6.5605,2.6036,0.0092,True
2,1,begin_checkout,begin_checkout,session,3784,45362,0.0834,4021,45193,0.0890,6.6606,2.9788,0.0029,True
3,1,new_accounts,new_accounts,session,3823,45362,0.0843,3681,45193,0.0815,-3.3543,-1.5429,0.1229,False
4,2,add_payment_info,add_payment_info,session,2344,50637,0.0463,2409,50244,0.0479,3.5769,1.2410,0.2146,False


In [ ]:
# --------------------------------------------
# Format numeric values
# --------------------------------------------

results_df["conversion_rate_control"] = results_df["conversion_rate_control"].round(6)
results_df["conversion_rate_variant"] = results_df["conversion_rate_variant"].round(6)

results_df["metric_change"] = results_df["metric_change"].round(3)

results_df["z_stat"] = results_df["z_stat"].round(3)
results_df["p_value"] = results_df["p_value"].round(6)

display(results_df)

,test_number,metric,numerator_event,denominator_event,numerator_control,denominator_control,conversion_rate_control,numerator_variant,denominator_variant,conversion_rate_variant,metric_change,z_stat,p_value,significant
0,1,add_payment_info,add_payment_info,session,1988,45362,0.0438,2229,45193,0.0493,12.5420,3.9250,0.0001,True
1,1,add_shipping_info,add_shipping_info,session,3034,45362,0.0669,3221,45193,0.0713,6.5600,2.6040,0.0092,True
2,1,begin_checkout,begin_checkout,session,3784,45362,0.0834,4021,45193,0.0890,6.6610,2.9790,0.0029,True
3,1,new_accounts,new_accounts,session,3823,45362,0.0843,3681,45193,0.0815,-3.3540,-1.5430,0.1229,False
4,2,add_payment_info,add_payment_info,session,2344,50637,0.0463,2409,50244,0.0479,3.5770,1.2410,0.2146,False
5,2,add_shipping_info,add_shipping_info,session,3480,50637,0.0687,3510,50244,0.0699,1.6510,0.7100,0.4780,False
6,2,begin_checkout,begin_checkout,session,4262,50637,0.0842,4313,50244,0.0858,1.9880,0.9530,0.3406,False
7,2,new_accounts,new_accounts,session,4165,50637,0.0823,4184,50244,0.0833,1.2420,0.5890,0.5560,False
8,3,add_payment_info,add_payment_info,session,3623,70047,0.0517,3697,70439,0.0525,1.4750,0.6430,0.5201,False
9,3,add_shipping_info,add_shipping_info,session,5298,70047,0.0756,5188,70439,0.0737,-2.6210,-1.4140,0.1574,False


## **9. Export Dataset for Dashboard**

To visualize the results in Tableau Public, we export the final results table as a CSV file.

This dataset will be used to create an interactive dashboard showing conversion rates, metric changes, and statistical significance across experiments.

In [ ]:
# --------------------------------------------
# Export results to CSV
# --------------------------------------------

results_df.to_csv("ab_test_results.csv", index=False)

print("File exported: ab_test_results.csv")

File exported: ab_test_results.csv


## **10. Tableau Dashboard**

An interactive dashboard was created in Tableau Public to visualize the results of the A/B testing analysis.

The dashboard focuses on comparing control and variant performance across key conversion metrics.

The dashboard includes:

• Conversion Rate Comparison (Control vs Variant)  
• Metric Change (%)  
• P-value Visualization  
• Statistical Significance Overview

Tableau Public Link:

https://public.tableau.com/shared/2YFHPYTHJ?:display_count=n&:origin=viz_share_link

Link to the result of the request:

https://drive.google.com/file/d/10nMYH8Bx0WRJ5ok5IMje1cVSpNFUvH5G/view?usp=sharing